# Multi-location bloom — do circles appear everywhere algorithmic, or only in W_U?

**Depends on `training_slowmo.ipynb`.** This notebook reads `snapshots.pt` (9 GB) produced by the slow-mo training run. Run that notebook first; by default its output lands in `./outputs/slowmo/snapshots.pt`.

**Standalone.** No Google Drive required. Runs anywhere a GPU is available. CPU works too, but extracting five locations across ten frames is slow without one.

---

**What it tests.** Clean Fourier neurons in FFN1 are memorized basis-function atoms. The algorithm lives in the *circle structure* — in `W_U`, in composed projections like `W_E @ V`, and in the *activation manifold* (residual stream grouped by `la − lb`). If this is right, circles should bloom *everywhere algorithmically meaningful*, not just at the output projection.

**Method.** For 10 selected steps spanning the trajectory, project five different objects onto their respective *final-state* fixed Fourier planes at frequency k=11:

1. **W_U** (output projection) — known to bloom.
2. **W_E @ V (L1)** — composition that delivers Fourier into residual stream via attention values.
3. **Residual stream at pre_head, grouped by (la − lb)** — averaged activation per dlog-difference. The actual *algorithmic manifold* the model uses to decide outputs.
4. **FFN1-mid at eq position, grouped by (la − lb)** — the Fourier filter outputs themselves.
5. **W_E** (control) — known to NOT have clean circles at final. Should NOT bloom.

Display: 5×10 grid (locations × time). Each row uses its own *final* plane as fixed coordinate system.

**Outputs (`./outputs/multiloc/`):**
- `multiloc_diffusion.png` — 5×10 grid showing per-step crystallization across locations
- `bloom_curves.png` — CV trajectories for each location overlaid
- `summary.json` — CV per frame per location

In [ ]:
# Cell 1 — Setup
# If running in Colab and you want I/O on Drive, uncomment:
#   from google.colab import drive
#   drive.mount('/content/drive')
# and point SLOMO_DIR / OUT_DIR at your Drive paths.

import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, json, time, os, gc
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.use_deterministic_algorithms(True, warn_only=True)

# Reads snapshots.pt produced by training_slowmo.ipynb.
# Run that notebook first; its default output is ./outputs/slowmo/snapshots.pt
SLOMO_DIR = Path('./outputs/slowmo')
OUT_DIR = Path('./outputs/multiloc')
OUT_DIR.mkdir(parents=True, exist_ok=True)

P = 97; PM1 = 96
D_MODEL = 384; D_FFN = 4*D_MODEL; N_HEADS = 12
BATCH = 1024

BG, FG, GRID, MUTED = '#FAFAF7', '#2A2A2A', '#E5E3DD', '#6E6E6E'
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG, 'savefig.facecolor': BG,
    'axes.edgecolor': FG, 'axes.linewidth': 0.5,
    'axes.labelcolor': FG, 'axes.titlecolor': FG,
    'xtick.color': FG, 'ytick.color': FG, 'text.color': FG,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica Neue', 'Helvetica', 'DejaVu Sans'],
    'font.size': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
})
TOKEN_CMAP = plt.cm.twilight
token_colors = TOKEN_CMAP(np.arange(PM1) / PM1)

def is_gen(g, p):
    seen = set(); x = 1
    for _ in range(p-1):
        x = (x*g) % p
        if x in seen: return False
        seen.add(x)
    return len(seen) == p-1
for g in range(2, P):
    if is_gen(g, P): break
exp_table = np.zeros(PM1, dtype=np.int64); x = 1
for t in range(PM1): exp_table[t] = x; x = (x*g) % P
dlog = np.zeros(P, dtype=np.int64)
for t in range(PM1): dlog[exp_table[t]] = t

snap_path = SLOMO_DIR / 'snapshots.pt'
if not snap_path.exists():
    raise FileNotFoundError(
        f'Snapshots not found at {snap_path.resolve()}.\n'
        f'Run training_slowmo.ipynb first to generate them, '
        f'or point SLOMO_DIR at an existing snapshots.pt.'
    )
print(f'Loading snapshots from {snap_path.resolve()} (9 GB, takes a minute)...')
snapshots = torch.load(snap_path, map_location='cpu', weights_only=False)
print(f'Loaded {len(snapshots)} snapshots, g={g}')

In [ ]:
# Cell 2 — Model + data + utilities
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-8):
        super().__init__(); self.scale = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.scale

class GrokBlock(nn.Module):
    def __init__(self, d, nh, dropout=0.2):
        super().__init__()
        self.norm1 = RMSNorm(d); self.attn = nn.MultiheadAttention(d, nh, dropout=dropout, batch_first=True)
        self.norm2 = RMSNorm(d); self.w1 = nn.Linear(d, 4*d, bias=False)
        self.w2 = nn.Linear(4*d, d, bias=False); self.w3 = nn.Linear(d, 4*d, bias=False)
        self.dropout = nn.Dropout(dropout)
        self._ffn_mid = None
    def forward(self, x):
        h = self.norm1(x); o, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.dropout(o)
        h2 = self.norm2(x); gate = F.silu(self.w1(h2))
        ffn_mid = gate * self.w3(h2)
        self._ffn_mid = ffn_mid[:, -1, :].detach()
        return x + self.dropout(self.w2(ffn_mid))

class GrokModel(nn.Module):
    def __init__(self, p=97, d=384, nh=12):
        super().__init__()
        self.tok_emb = nn.Embedding(p+2, d); self.pos_emb = nn.Embedding(4, d)
        self.blocks = nn.ModuleList([GrokBlock(d, nh) for _ in range(2)])
        self.norm_f = RMSNorm(d); self.head = nn.Linear(d, p, bias=False); self.p = p
    def forward(self, a, b, capture_residual=False):
        B = a.size(0)
        op = torch.full((B,), self.p, device=a.device, dtype=torch.long)
        eq = torch.full((B,), self.p+1, device=a.device, dtype=torch.long)
        tok = torch.stack([a, op, b, eq], dim=1)
        pos = torch.arange(4, device=a.device).unsqueeze(0).expand(B, -1)
        x = self.tok_emb(tok) + self.pos_emb(pos)
        for bl in self.blocks: x = bl(x)
        h = self.norm_f(x)[:, -1, :]
        if capture_residual: return self.head(h), h
        return self.head(h)

all_a, all_b = [], []
for a in range(P):
    for b in range(1, P):
        all_a.append(a); all_b.append(b)
all_a = torch.tensor(all_a, device=device)
all_b = torch.tensor(all_b, device=device)
diff_arr = np.array([
    -1 if int(all_a[i].item()) == 0 else (int(dlog[int(all_a[i].item())]) - int(dlog[int(all_b[i].item())])) % PM1
    for i in range(len(all_a))
])

def fourier_decompose_dlog(M):
    p = M.shape[0]; K = p//2 + 1
    i = np.arange(p)
    A = np.zeros((K, M.shape[1])); B = np.zeros((K, M.shape[1]))
    for k in range(K):
        c = np.cos(2*np.pi*k*i/p); s = np.sin(2*np.pi*k*i/p)
        norm = p/2 if k != 0 and k != p//2 else p
        A[k] = c @ M / norm; B[k] = s @ M / norm
    return A, B, (A**2).sum(1) + (B**2).sum(1)

def build_orthonormal_2d(a, b):
    u = a / (np.linalg.norm(a) + 1e-12)
    v = b - (b @ u) * u
    v = v / (np.linalg.norm(v) + 1e-12)
    return np.stack([u, v], axis=1)

def cv_radii(xy):
    cx, cy = xy.mean(0)
    r = np.sqrt((xy[:,0]-cx)**2 + (xy[:,1]-cy)**2)
    return r.std() / (r.mean() + 1e-12)

print('Model + utilities ready')

In [ ]:
# Cell 3 — Extractors: 5 locations from a state_dict
# Each returns a [96, d] matrix.

@torch.no_grad()
def extract_locations(state_dict):
    sd_dev = {k: v.float().to(device) for k, v in state_dict.items()}
    m = GrokModel(P, D_MODEL, N_HEADS).to(device)
    m.load_state_dict(sd_dev, strict=False); m.eval()
    # 1. W_U: dlog reindexed
    wu = sd_dev['head.weight'].cpu().numpy()  # [P, d]
    wu_dlog = wu[exp_table]
    # 2. W_E @ V (L1): pass W_E through layer-1 norm and attn.V projection
    in_w = sd_dev['blocks.1.attn.in_proj_weight'].cpu().numpy()
    in_b = sd_dev.get('blocks.1.attn.in_proj_bias')
    norm1_scale = sd_dev['blocks.1.norm1.scale'].cpu().numpy()
    Wv = in_w[2*D_MODEL:]
    bv = in_b[2*D_MODEL:].cpu().numpy() if in_b is not None else 0
    we = sd_dev['tok_emb.weight'][:P].cpu().numpy()
    eps = 1e-8
    h_norm = we * (1.0 / np.sqrt((we**2).mean(axis=-1, keepdims=True) + eps)) * norm1_scale
    we_v = (h_norm @ Wv.T + bv)[exp_table]  # [96, d]
    # 5. W_E (control, raw)
    we_dlog = sd_dev['tok_emb.weight'][:P].cpu().numpy()[exp_table]
    # 3. residual @ pre_head, per (la-lb)
    # 4. FFN1-mid, per (la-lb)
    res_out = np.zeros((PM1, D_MODEL)); res_counts = np.zeros(PM1)
    ffn_out = np.zeros((PM1, D_FFN)); ffn_counts = np.zeros(PM1)
    for i in range(0, len(all_a), BATCH):
        ab = all_a[i:i+BATCH]; bb = all_b[i:i+BATCH]
        _, h = m(ab, bb, capture_residual=True)
        h_np = h.cpu().numpy()
        ffn = m.blocks[1]._ffn_mid.cpu().numpy()
        for j in range(len(ab)):
            di = diff_arr[i + j]
            if di < 0: continue
            res_out[di] += h_np[j]; res_counts[di] += 1
            ffn_out[di] += ffn[j]; ffn_counts[di] += 1
    res_out /= np.maximum(res_counts[:, None], 1)
    ffn_out /= np.maximum(ffn_counts[:, None], 1)
    del m; gc.collect(); torch.cuda.empty_cache()
    return {
        'W_U': wu_dlog,                           # [96, 384]
        'W_E_V_L1': we_v,                         # [96, 384]
        'residual_per_diff': res_out,             # [96, 384]
        'FFN1_per_diff': ffn_out,                 # [96, 1536]
        'W_E_control': we_dlog,                   # [96, 384]
    }

# Compute FINAL state's locations and build fixed planes
K_TARGET = 11
print('Computing final state locations + building fixed planes...')
t0 = time.time()
final_loc = extract_locations(snapshots[-1]['state_dict'])
print(f'  done in {time.time()-t0:.1f}s')

fixed_planes = {}
for name, M in final_loc.items():
    A, B, _ = fourier_decompose_dlog(M)
    fixed_planes[name] = build_orthonormal_2d(A[K_TARGET], B[K_TARGET])
    # Sanity: CV in this final-fixed plane should be small for algorithmic locations
    xy = M @ fixed_planes[name]
    print(f'  {name:<22}: final CV at k={K_TARGET} = {cv_radii(xy):.3f}')

In [ ]:
# Cell 4 — Process selected frames
all_steps = [(-1 if s['step'] == 'pre' else int(s['step'])) for s in snapshots]
all_steps_arr = np.array(all_steps)

target_frames = [0, 50, 100, 200, 300, 400, 500, 700, 900, all_steps_arr.max()]
frame_indices = []
for t in target_frames:
    idx = int(np.argmin(np.abs(all_steps_arr - t)))
    frame_indices.append(idx)
frame_indices = sorted(set(frame_indices))
print(f'Frames at steps: {[int(all_steps_arr[i]) for i in frame_indices]}')

# For each frame: extract all 5 locations, project on fixed planes, record xy and CV
frame_records = []
t0 = time.time()
for fi in frame_indices:
    locs = extract_locations(snapshots[fi]['state_dict'])
    rec = {'step': snapshots[fi]['step'], 'test_acc': snapshots[fi]['test_acc'], 'locs': {}}
    for name, M in locs.items():
        xy = M @ fixed_planes[name]
        cv = cv_radii(xy)
        rec['locs'][name] = {'xy': xy, 'cv': float(cv)}
    frame_records.append(rec)
    print(f'  step {rec["step"]}: acc={rec["test_acc"]:.3f}, '
          + ', '.join(f'{n}={rec["locs"][n]["cv"]:.2f}' for n in ['W_U','W_E_V_L1','residual_per_diff','FFN1_per_diff','W_E_control']))
print(f'\nTotal: {time.time()-t0:.1f}s')

In [ ]:
# Cell 5 — Money shot: 5×N grid (locations × time)
LOC_ORDER = ['W_U', 'W_E_V_L1', 'residual_per_diff', 'FFN1_per_diff', 'W_E_control']
LOC_LABELS = {
    'W_U': 'W_U (head)',
    'W_E_V_L1': 'W_E @ V (L1)',
    'residual_per_diff': 'residual @ pre_head\nper (la−lb)',
    'FFN1_per_diff': 'FFN1-mid\nper (la−lb)',
    'W_E_control': 'W_E (control)',
}
n_rows = len(LOC_ORDER); n_cols = len(frame_records)

# Build per-row axis bounds (use final-state extent as reference)
row_bounds = {}
for name in LOC_ORDER:
    all_xys = np.concatenate([rec['locs'][name]['xy'] for rec in frame_records])
    qx = np.percentile(np.abs(all_xys[:,0] - all_xys[:,0].mean()), 99)
    qy = np.percentile(np.abs(all_xys[:,1] - all_xys[:,1].mean()), 99)
    row_bounds[name] = max(qx, qy) * 1.15

fig, axes = plt.subplots(n_rows, n_cols, figsize=(min(12, 1.2 * n_cols), 1.2 * n_rows))
for r, name in enumerate(LOC_ORDER):
    bound = row_bounds[name]
    for c, rec in enumerate(frame_records):
        ax = axes[r, c] if n_rows > 1 else axes[c]
        xy = rec['locs'][name]['xy']
        cv = rec['locs'][name]['cv']
        cx, cy = xy.mean(0)
        ax.scatter(xy[:,0] - cx, xy[:,1] - cy, c=token_colors, s=6,
                   edgecolors='white', linewidths=0.15, zorder=3)
        ax.set_aspect('equal')
        ax.set_xlim(-bound, bound); ax.set_ylim(-bound, bound)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for sp in ax.spines.values(): sp.set_visible(False)
        # subtle reference circle at mean radius
        r_mean = np.sqrt(((xy - xy.mean(0))**2).sum(1)).mean()
        theta = np.linspace(0, 2*np.pi, 100)
        ax.plot(r_mean*np.cos(theta), r_mean*np.sin(theta), color=GRID, lw=0.4, alpha=0.7, zorder=1)
        # CV text
        ax.text(0.05, 0.95, f'{cv:.2f}', transform=ax.transAxes, fontsize=6,
                va='top', ha='left', color=MUTED, family='monospace')
        # Top labels (step) on first row only
        if r == 0:
            step_lbl = rec['step']
            step_str = 'init' if step_lbl == 'pre' else f"{step_lbl}"
            acc_str = f"{rec['test_acc']:.2f}"
            ax.set_title(f'{step_str}\nacc={acc_str}', fontsize=7.5, pad=2)
        # Row labels on first column
        if c == 0:
            ax.set_ylabel(LOC_LABELS[name], fontsize=8.5, rotation=0,
                          ha='right', va='center', labelpad=12)

fig.suptitle(f'Multi-location diffusion at k={K_TARGET}: do circles bloom everywhere or only in W_U?\n'
             f'Each row projects onto its own final-fixed Fourier plane. CV in corners.',
             fontsize=10, y=1.005)
plt.tight_layout()
plt.savefig(OUT_DIR / 'multiloc_diffusion.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 6 — Bloom curves: CV over time per location
fig, ax = plt.subplots(figsize=(11, 4.5))
step_num = np.array([(-1 if r['step'] == 'pre' else int(r['step'])) for r in frame_records])
colors = {
    'W_U': '#3A6E8C',
    'W_E_V_L1': '#D45A2A',
    'residual_per_diff': '#3A8C3A',
    'FFN1_per_diff': '#7A4DA8',
    'W_E_control': '#6E6E6E',
}
for name in LOC_ORDER:
    cvs = [rec['locs'][name]['cv'] for rec in frame_records]
    ls = ':' if name == 'W_E_control' else '-'
    ax.plot(step_num, cvs, ls, color=colors[name], lw=1.8, marker='o', markersize=4,
            label=LOC_LABELS[name].replace('\n', ' '), alpha=0.9)
ax.axhline(0.15, color='#3A8C3A', ls='--', lw=0.6, alpha=0.6, label='clean threshold (0.15)')
ax.set_xlabel('gradient step'); ax.set_ylabel('circle CV (lower = cleaner)')
ax.set_title(f'Per-location Fourier circle quality over training (k={K_TARGET})')
ax.legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.savefig(OUT_DIR / 'bloom_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 7 — Summary
summary = {
    'config': {'P': P, 'PM1': PM1, 'g': int(g), 'K_TARGET': K_TARGET},
    'final_cvs_at_k': {name: float(cv_radii(M @ fixed_planes[name])) for name, M in final_loc.items()},
    'frames': [{
        'step': r['step'],
        'test_acc': float(r['test_acc']),
        'cvs': {name: float(r['locs'][name]['cv']) for name in LOC_ORDER},
    } for r in frame_records],
}
with open(OUT_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Saved {OUT_DIR / "summary.json"}')
print(f'\nFinal CVs at k={K_TARGET}:')
for name, cv in summary['final_cvs_at_k'].items():
    print(f'  {name:<22}: {cv:.3f}')

## How to read Plan 1f

**`multiloc_diffusion.png`.** Five rows of crystallization trajectories. Each cell shows the 96 dlog-grouped points (tokens or (la−lb)-classes) at one training step, projected into that row's own final-fixed Fourier plane. CV in each corner.

**Interpretation matrix:**
- If `residual_per_diff` and `FFN1_per_diff` rows bloom alongside `W_U` and `W_E_V_L1` → the algorithm is genuinely *distributed* across the model. Circles are everywhere there's algorithmic computation. The user's reframing is confirmed.
- If only `W_U` and `W_E_V_L1` bloom while activation rows stay diffuse → algorithmic structure is in *weights only*, not in activation geometry. Algorithm "lives in the parameters," not in the computed manifolds.
- If everything blooms simultaneously → strong co-alignment (consistent with Plan 1e finding).
- If they bloom at different rates → there's a *causal ordering* across locations. Whichever blooms first is the algorithmic "prime mover."

**`bloom_curves.png`.** Five CV trajectories on one plot. Quick visual: do they all drop together, or in stages?

## What this tests for the user's hypothesis

The user's claim: clean FFN1 neurons are memorized basis atoms; the algorithm is circle structure everywhere algorithmically meaningful, not just in W_U.

Direct test: do circles appear in the activation manifolds (residual stream and FFN1-mid grouped by la−lb)? If yes — and they bloom synchronously with W_U — the user is correct that algorithm is distributed circle structure, with FFN1 neurons as one component among many.

If activation circles don't appear (CV stays at random level even at final): clean FFN1 neurons + W_U circles are sufficient on their own, and the activation manifold view is not the dominant lens.

Either result reshapes the mechanistic picture.